# Market Differentiated Bars

## Overview

This notebook demonstrates the public functions in `market_differentiated_bars` using dollar-bar close prices built from a compact synthetic trade stream.
- Problem: integer differencing can remove more memory than is needed to make a price series stationary.
- Approach: compare fractional-differencing weights and fixed-width transformed prices across differencing orders.
- Weight Inspection: It plots the fractional-differencing weights described in AFML Figure 5.1.
- Fractional Differencing: It compares the original close path with a fixed-width fractional difference as in AFML Figure 5.4.
- Minimum-FFD Diagnostic: It measures the stationarity-memory tradeoff shown in AFML Figure 5.5.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
from src.data_preprocessing.market_differentiated_bars import (
    fractional_difference,
    fractional_difference_fixed_width,
    get_weights,
    get_weights_fixed_width,
    plot_min_ffd,
    plot_weights,
)
from src.data_preprocessing.market_structured_bars import get_dollar_bars

## Define Synthetic Trades

This cell defines the reproducible intraday trade stream and resamples it into dollar bars.
- close_frame is the single-column input required by the fractional-differencing functions.
- target_bars_per_day sets the dollar-bar threshold from the synthetic trading-day count.


In [ ]:
rng = np.random.default_rng(42)
trading_days = pd.bdate_range("2024-01-02", periods=20, tz="UTC")
trades_per_day = 48
minutes_from_open = np.tile(np.arange(trades_per_day) * 5 + 14 * 60 + 30, len(trading_days))
timestamps = trading_days.repeat(trades_per_day) + pd.to_timedelta(minutes_from_open, unit="m")
prices = 180.0 * np.exp(np.cumsum(rng.normal(0.00005, 0.0015, len(timestamps))))
sizes = rng.lognormal(3.7, 0.55, len(timestamps)).round().astype(int)
trades = pd.DataFrame({"timestamp": timestamps, "symbol": "AAPL", "price": prices, "size": sizes})
notional = trades["price"].astype(float) * trades["size"].astype(float)
target_bars_per_day = 24
trading_days = len(trading_days)
target_num_bars = max(1, trading_days * target_bars_per_day)
dollar_threshold = float(notional.sum() / target_num_bars)
dollar_bars = get_dollar_bars(trades, threshold=dollar_threshold).ohlcv
close_frame = dollar_bars[["close"]].astype(float)

print("source: synthetic intraday trade stream")
print(f"trading_days: {trading_days}")
print(f"target_bars_per_day: {target_bars_per_day}")
print(f"num_dollar_bars: {len(close_frame):,}")
close_frame.head()

## Inspect Fractional-Differencing Weights

This cell inspects expanding-window and fixed-width fractional-differencing weights.
- get_weights returns the expanding-window weight sequence.
- get_weights_fixed_width drops weights below the configured threshold.
- plot_weights renders the weight curves corresponding to AFML Figure 5.1.


In [ ]:
weight_example = pd.DataFrame(get_weights(d=0.4, size=8).T, columns=[f"w_{i}" for i in range(8)])
weight_example

This cell inspects the fixed-width weight sequence after applying the cutoff threshold.


In [ ]:
fixed_width_weights = get_weights_fixed_width(d=0.4, thres=0.05)
fixed_width_weight_example = pd.DataFrame(
    fixed_width_weights.T,
    columns=[f"w_{i}" for i in range(fixed_width_weights.shape[0])],
)
fixed_width_weight_example

This cell plots weight curves across fractional-differencing orders.


In [ ]:
plot_weights(d_range=[0.0, 1.0], n_plots=5, size=10)

## Apply Fractional Differencing

This cell applies expanding-window and fixed-width fractional differencing to the same close series.
- The row-count table compares the sample loss of the two methods.
- The plot follows AFML Figure 5.4 by comparing the original dollar-bar close with the fixed-width transform.


In [ ]:
ffd_expanding = fractional_difference(close_frame, d=0.4, thres=0.01)
ffd_fixed = fractional_difference_fixed_width(close_frame, d=0.4, thres=1e-3)

pd.Series(
    {
        "original_rows": len(close_frame),
        "expanding_rows": len(ffd_expanding),
        "fixed_width_rows": len(ffd_fixed),
    },
    name="num_rows",
).to_frame()

This cell compares the original close path with the fixed-width transformed series.


In [ ]:
comparison = close_frame.rename(columns={"close": "original_close"}).join(
    ffd_fixed.rename(columns={"close": "fixed_width_ffd"}),
    how="left",
)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
comparison["original_close"].plot(ax=axes[0], color="tab:blue", title="Original dollar-bar close")
comparison["fixed_width_ffd"].plot(ax=axes[1], color="tab:green", title="Fixed-width fractional difference")

for ax in axes:
    ax.grid(alpha=0.25)
    ax.set_xlabel("time")

fig.suptitle("AFML Figure 5.4: Fixed-width fractional differentiation")
fig.tight_layout()
comparison.dropna().head()


## Minimum-FFD Diagnostic

This cell evaluates stationarity and memory retention across fractional-differencing orders.
- plot_min_ffd reports the ADF statistic and correlation with the original log-close series.
- The diagnostic plot follows AFML Figure 5.5.


In [ ]:
min_ffd_diagnostic = plot_min_ffd(close_frame, thres=0.01)
min_ffd_diagnostic